In [ ]:
import os
import json

def generate_file_tree(root_dir):
    stack = [root_dir]  # 模拟递归的栈
    file_tree = {"name": os.path.basename(root_dir), "type": "directory", "children": []}
    dir_map = {root_dir: file_tree}  # 记录已访问的目录对应的树节点

    while stack:
        current_dir = stack.pop()
        current_node = dir_map[current_dir]  # 获取当前目录对应的树节点
        
        try:
            items = os.listdir(current_dir)  # 只能读取当前文件夹的文件
        except PermissionError:
            continue  # 忽略无权限访问的文件夹

        for item in items:
            item_path = os.path.join(current_dir, item)
            node = {"name": item, "type": "directory" if os.path.isdir(item_path) else "file"}
            current_node.setdefault("children", []).append(node)

            if os.path.isdir(item_path):  # 如果是文件夹，加入待遍历列表
                stack.append(item_path)
                dir_map[item_path] = node  # 记录该文件夹的树节点

    return file_tree

# 指定目录
folder_path = "your_directory_path_here"

# 生成文件树
file_tree = generate_file_tree(folder_path)

# 保存 JSON
json_output_path = "file_tree.json"
with open(json_output_path, "w", encoding="utf-8") as f:
    json.dump(file_tree, f, indent=4, ensure_ascii=False)

print(f"文件树已保存至 {json_output_path}")


In [ ]:
import os
import json
from abc import ABC, abstractmethod

# 抽象基类：代表文件系统中的一个节点（文件/文件夹）
class Node(ABC):
    def __init__(self, name):
        self.name = name

    @abstractmethod
    def to_dict(self):
        pass

# 文件类
class FileNode(Node):
    def to_dict(self):
        return {"name": self.name, "type": "file"}

# 目录类
class DirectoryNode(Node):
    def __init__(self, name):
        super().__init__(name)
        self.children = []

    def add_child(self, child):
        self.children.append(child)

    def to_dict(self):
        return {"name": self.name, "type": "directory", "children": [child.to_dict() for child in self.children]}

# 工厂类：用于创建文件或文件夹节点
class NodeFactory:
    @staticmethod
    def create_node(path):
        name = os.path.basename(path)
        if os.path.isdir(path):
            return DirectoryNode(name)
        else:
            return FileNode(name)

# 文件树构造器
class FileTreeBuilder:
    def __init__(self, root_dir):
        self.root_dir = root_dir

    def build_tree(self):
        stack = [self.root_dir]  # 目录栈
        root_node = NodeFactory.create_node(self.root_dir)  # 创建根目录节点
        dir_map = {self.root_dir: root_node}  # 记录目录对应的节点
        
        while stack:
            current_dir = stack.pop()
            current_node = dir_map[current_dir]

            try:
                items = os.listdir(current_dir)  # 只能读取当前文件夹内容
            except PermissionError:
                continue  # 忽略无权限访问的文件夹

            for item in items:
                item_path = os.path.join(current_dir, item)
                node = NodeFactory.create_node(item_path)  # 使用工厂创建节点
                current_node.add_child(node) if isinstance(current_node, DirectoryNode) else None

                if isinstance(node, DirectoryNode):  # 如果是文件夹，加入待遍历列表
                    stack.append(item_path)
                    dir_map[item_path] = node

        return root_node

# 运行代码
folder_path = "your_directory_path_here"  # 替换为你的目录路径

# 生成文件树
builder = FileTreeBuilder(folder_path)
file_tree = builder.build_tree()

# 保存 JSON
json_output_path = "file_tree.json"
with open(json_output_path, "w", encoding="utf-8") as f:
    json.dump(file_tree.to_dict(), f, indent=4, ensure_ascii=False)

print(f"文件树已保存至 {json_output_path}")


In [ ]:
{
    "name": "root_folder",
    "type": "directory",
    "children": [
        {
            "name": "subfolder",
            "type": "directory",
            "children": [
                {
                    "name": "file1.txt",
                    "type": "file"
                }
            ]
        },
        {
            "name": "file2.txt",
            "type": "file"
        }
    ]
}


In [ ]:
import os
import json
from abc import ABC, abstractmethod

# 实现层接口（Bridge Implementor）
class NodeImplementor(ABC):
    def __init__(self, name):
        self.name = name

    @abstractmethod
    def to_dict(self):
        pass

# 文件实现（Concrete Implementor）
class FileNodeImplementor(NodeImplementor):
    def to_dict(self):
        return {"name": self.name, "type": "file"}

# 目录实现（Concrete Implementor）
class DirectoryNodeImplementor(NodeImplementor):
    def __init__(self, name):
        super().__init__(name)
        self.children = []

    def add_child(self, child):
        self.children.append(child)

    def to_dict(self):
        return {"name": self.name, "type": "directory", "children": [child.to_dict() for child in self.children]}

# 抽象层（Abstraction）
class Node(ABC):
    def __init__(self, implementor):
        self.implementor = implementor

    @abstractmethod
    def to_dict(self):
        pass

# 文件节点（Refined Abstraction）
class FileNode(Node):
    def __init__(self, name):
        super().__init__(FileNodeImplementor(name))

    def to_dict(self):
        return self.implementor.to_dict()

# 目录节点（Refined Abstraction）
class DirectoryNode(Node):
    def __init__(self, name):
        super().__init__(DirectoryNodeImplementor(name))

    def add_child(self, child):
        self.implementor.add_child(child.implementor)

    def to_dict(self):
        return self.implementor.to_dict()

# 文件树构造器
class FileTreeBuilder:
    def __init__(self, root_dir):
        self.root_dir = root_dir

    def build_tree(self):
        stack = [self.root_dir]
        root_node = DirectoryNode(os.path.basename(self.root_dir))
        dir_map = {self.root_dir: root_node}

        while stack:
            current_dir = stack.pop()
            current_node = dir_map[current_dir]

            try:
                items = os.listdir(current_dir)
            except PermissionError:
                continue

            for item in items:
                item_path = os.path.join(current_dir, item)
                if os.path.isdir(item_path):
                    node = DirectoryNode(item)
                    stack.append(item_path)
                    dir_map[item_path] = node
                else:
                    node = FileNode(item)

                current_node.add_child(node)

        return root_node

# 运行代码
folder_path = "your_directory_path_here"

# 生成文件树
builder = FileTreeBuilder(folder_path)
file_tree = builder.build_tree()

# 保存 JSON
json_output_path = "file_tree.json"
with open(json_output_path, "w", encoding="utf-8") as f:
    json.dump(file_tree.to_dict(), f, indent=4, ensure_ascii=False)

print(f"文件树已保存至 {json_output_path}")
